# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `ppangya/ppangya-brain` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/gemma-4-E2B-it`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `ppangya/pnk-brain-v1` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0003 · steps 112 · seq 1024 · linear · 양자화 q4_k_m (데이터 56개)


In [11]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [12]:
from huggingface_hub import notebook_login
notebook_login()


In [13]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


==((====))==  Unsloth 2026.6.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/301k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

✅ 베이스 모델 로딩 완료


In [23]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


KeyboardInterrupt: 

## 📦 단기 지식 데이터셋 불러오기 (conversations Q&A)
Connect AI 앱이 업로드한 데이터셋. 각 행 = `{conversations:[{user},{assistant}]}`


In [13]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
ds = load_dataset("ppangya/parknk21c-longset", data_files="connect-ai-brain.jsonl", split="train", token=True)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])

connect-ai-brain.jsonl:   0%|          | 0.00/13.7k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

데이터 개수: 27
<|turn>user
미국 학위만으로는 취업이 어려운 시대에 실제 직무 역량을 증명할 수 있는 방법은 무엇인가요<turn|>
<|turn>model
미국 학사·석사 학위를 10년·큰 비용을 들여 땄지만, 지금은 그 졸업장만으로 취업이 어렵다. 인턴 요건조차 "분야 상위 1%·수학 올림피아드 입상" 수준으로 비현실적으로 높아졌다.<turn|>



In [25]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 50, learning_rate = 0.0003,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)

NameError: name 'ds' is not defined

In [15]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 모델/버전마다 다름(<|turn> vs <start_of_turn>) → 실제 텍스트에서 자동 감지
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
_im = "<|turn>user\n" if "<|turn>user" in _t else "<start_of_turn>user\n"
_rm = "<|turn>model\n" if "<|turn>model" in _t else "<start_of_turn>model\n"
trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 학습 준비 완료")


Map (num_proc=6):   0%|          | 0/27 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/27 [00:00<?, ? examples/s]

✅ 마스킹 마커 자동감지: <|turn>model — 학습 준비 완료


In [ ]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [7]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
    inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to("cuda")
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


NameError: name 'model' is not defined

## 💾 GGUF로 저장 (Connect AI 내장 엔진용)
테스트가 만족스러우면 업로드! (맨 앞에서 로그인했으니 바로 됩니다)


In [10]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
print("메모리 정리 완료 — GGUF 변환 시작")
# 내 모델 = 장기 기억. q4_k_m GGUF 로 저장 + HF 업로드
model.push_to_hub_gguf("ppangya/pnk-brain-v1", tokenizer, quantization_method="q4_k_m", token=True)
print("✅ 완료! Connect AI 앱 → 🤖 내 AI 팀 → HuggingFace에서 받기 → \"ppangya/pnk-brain-v1\" 검색해서 내려받으면 내 모델로 바로 사용 (LM Studio 불필요)")


메모리 정리 완료 — GGUF 변환 시작


NameError: name 'model' is not defined

## 💽 디스크 공간 확인

GGUF 변환이 계속 멈춘다면, Colab 인스턴스의 디스크 공간이 부족할 수 있습니다. 아래 코드를 실행하여 현재 디스크 사용량을 확인해 보세요.

In [3]:
import shutil

total, used, free = shutil.disk_usage("/")

print(f"총 디스크 공간: {total / (1024**3):.2f} GB")
print(f"사용 중인 공간: {used / (1024**3):.2f} GB")
print(f"남은 공간: {free / (1024**3):.2f} GB")

if free / (1024**3) < 10: # 남은 공간이 10GB 미만이면 경고
    print("⚠️ 경고: 남은 디스크 공간이 10GB 미만입니다. GGUF 변환에 문제가 발생할 수 있습니다.")
else:
    print("✅ 디스크 공간은 충분해 보입니다.")

총 디스크 공간: 112.64 GB
사용 중인 공간: 53.86 GB
남은 공간: 58.76 GB
✅ 디스크 공간은 충분해 보입니다.


만약 디스크 공간이 충분하다면, Hugging Face Hub 서버와의 연결이 불안정하거나 일시적인 네트워크 문제일 가능성이 있습니다. 이 경우, 잠시 기다렸다가 다시 시도해 보시는 것이 좋습니다.

## 📊 GGUF 변환 상세 로그 확인

`huggingface_hub` 캐시 문제 또는 다운로드 진행 상황에 대한 자세한 정보를 얻기 위해 로깅 레벨을 `DEBUG`로 설정합니다. 이 셀을 실행한 후, **GGUF 변환 셀(`lydMBGrnIBY5`)을 다시 실행**하면 더 많은 상세 로그를 확인할 수 있습니다.

In [1]:
import logging

# Hugging Face Hub 라이브러리의 로깅 레벨을 DEBUG로 설정
logging.basicConfig(level=logging.DEBUG)
logging.getLogger("huggingface_hub").setLevel(logging.DEBUG)

print("Hugging Face Hub 로깅 레벨이 DEBUG로 설정되었습니다. 이제 GGUF 변환 셀을 다시 실행해 보세요.")

Hugging Face Hub 로깅 레벨이 DEBUG로 설정되었습니다. 이제 GGUF 변환 셀을 다시 실행해 보세요.


## 🗑️ Hugging Face 캐시 삭제 (GGUF 변환 문제 해결 시도)

GGUF 변환이 계속 멈춘다면, Hugging Face 캐시 디렉토리가 손상되었거나 불완전한 파일이 남아있을 가능성이 있습니다. 아래 코드를 실행하여 캐시를 완전히 삭제한 후, **런타임을 재시작**하고 **모든 셀을 다시 실행**해 보세요.

In [2]:
import os
import shutil

# Hugging Face 캐시 디렉토리 경로
hf_cache_dir = os.path.expanduser("~/.cache/huggingface")

# 캐시 디렉토리 존재 여부 확인 및 삭제
if os.path.exists(hf_cache_dir):
    print(f"Hugging Face 캐시 디렉토리 삭제 중: {hf_cache_dir}")
    shutil.rmtree(hf_cache_dir)
    print("Hugging Face 캐시 디렉토리 삭제 완료.")
else:
    print("Hugging Face 캐시 디렉토리가 존재하지 않습니다.")

print("캐시 삭제 후, 런타임 > 런타임 다시 시작을 클릭하고 모든 셀을 처음부터 다시 실행해 주세요.")

Hugging Face 캐시 디렉토리 삭제 중: /root/.cache/huggingface
Hugging Face 캐시 디렉토리 삭제 완료.
캐시 삭제 후, 런타임 > 런타임 다시 시작을 클릭하고 모든 셀을 처음부터 다시 실행해 주세요.
